In [0]:
%sql
CREATE TABLE IF NOT EXISTS store_data AS 
SELECT
'{
   "store":{
      "fruit": [
        {"weight":8,"type":"apple"},
        {"weight":9,"type":"pear"}
      ],
      "basket":[
        [1,2,{"b":"y","a":"x"}],
        [3,4],
        [5,6]
      ],
      "book":[
        {
          "author":"Nigel Rees",
          "title":"Sayings of the Century",
          "category":"reference",
          "price":8.95
        },
        {
          "author":"Herman Melville",
          "title":"Moby Dick",
          "category":"fiction",
          "price":8.99,
          "isbn":"0-553-21311-3"
        },
        {
          "author":"J. R. R. Tolkien",
          "title":"The Lord of the Rings",
          "category":"fiction",
          "reader":[
            {"age":25,"name":"bob"},
            {"age":26,"name":"jack"}
          ],
          "price":22.99,
          "isbn":"0-395-19395-8"
        }
      ],
      "bicycle":{
        "price":19.95,
        "color":"red"
      }
    },
    "owner":"amy",
    "zip code":"94025",
    "fb:testid":"1234"
 }' as raw

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM store_data;

raw
"{ ""store"":{ ""fruit"": [ {""weight"":8,""type"":""apple""}, {""weight"":9,""type"":""pear""} ], ""basket"":[ [1,2,{""b"":""y"",""a"":""x""}], [3,4], [5,6] ], ""book"":[ { ""author"":""Nigel Rees"", ""title"":""Sayings of the Century"", ""category"":""reference"", ""price"":8.95 }, { ""author"":""Herman Melville"", ""title"":""Moby Dick"", ""category"":""fiction"", ""price"":8.99, ""isbn"":""0-553-21311-3"" }, { ""author"":""J. R. R. Tolkien"", ""title"":""The Lord of the Rings"", ""category"":""fiction"", ""reader"":[ {""age"":25,""name"":""bob""}, {""age"":26,""name"":""jack""} ], ""price"":22.99, ""isbn"":""0-395-19395-8"" } ], ""bicycle"":{ ""price"":19.95, ""color"":""red"" } }, ""owner"":""amy"", ""zip code"":""94025"", ""fb:testid"":""1234"" }"


# Extract a top-level column

In [0]:
%sql
SELECT raw:owner, RAW:OWNER, raw:['OWNER'] FROM store_data;

owner,OWNER,OWNER
amy,amy,null


In [0]:
%sql
SELECT raw:`zip code`, raw:`Zip Code`, raw:['fb:testid'] FROM store_data;

zip code,Zip Code,fb:testid
94025,94025,1234


## Extract nested fields

In [0]:
%sql
SELECT raw:store.bicycle FROM store_data;

bicycle
"{""price"":19.95,""color"":""red""}"


In [0]:
%sql
SELECT raw:store.fruit[0], raw:store.fruit[1] FROM store_data;

fruit,fruit
"{""weight"":8,""type"":""apple""}","{""weight"":9,""type"":""pear""}"


In [0]:
%sql
SELECT raw:store.book[*].isbn FROM store_data;

isbn
"[null,""0-553-21311-3"",""0-395-19395-8""]"


In [0]:
%sql
SELECT
    raw:store.basket[*],
    raw:store.basket[*][0] first_of_baskets,
    raw:store.basket[0][*] first_basket,
    raw:store.basket[*][*] all_elements_flattened,
    raw:store.basket[0][2].b subfield
FROM store_data;

basket,first_of_baskets,first_basket,all_elements_flattened,subfield
"[[1,2,{""b"":""y"",""a"":""x""}],[3,4],[5,6]]","[1,3,5]","[1,2,{""b"":""y"",""a"":""x""}]","[1,2,{""b"":""y"",""a"":""x""},3,4,5,6]",y


# Cast values

In [0]:
%sql
SELECT CAST(raw:store.bicycle.price AS DOUBLE) FROM store_data

price
19.95


In [0]:
%sql
SELECT from_json(raw:store.bicycle, 'price double, color string') bicycle 
FROM store_data

bicycle
"List(19.95, red)"


In [0]:
%sql
SELECT from_json(
  raw:store.bicycle, 
  schema_of_json(
    '{
      "price": 19.95,
      "color": "red"
    }'
  )
) bicycle FROM store_data;

bicycle
"List(red, 19.95)"


In [0]:
%sql
SELECT from_json(raw:store.basket[*], 'array<array<string>>') baskets 
FROM store_data;

baskets
"List(List(1, 2, {""b"":""y"",""a"":""x""}), List(3, 4), List(5, 6))"


# Referências
https://docs.databricks.com/aws/en/semi-structured/json